In [1]:
from pathlib import Path
import base64, gzip, hashlib, json, os, shutil, subprocess, sys

ASSET_ROOT = Path('/kaggle/working/uav_detr_offline_assets')
MODEL_ID = 'PekingU/rtdetr_r50vd_coco_o365'
MODEL_FOLDER = 'rtdetr_r50vd_coco_o365'
TRANSFORMERS_VERSION = '5.15.0'
COPY_VISDRONE_FROM = None    # e.g. '/kaggle/input/visdrone2019-det'

if ASSET_ROOT.exists():
    shutil.rmtree(ASSET_ROOT)
for directory in [f'models/{MODEL_FOLDER}', 'wheels', 'eval/detr']:
    (ASSET_ROOT / directory).mkdir(parents=True, exist_ok=True)
print('Asset root:', ASSET_ROOT)


Asset root: /kaggle/working/uav_detr_offline_assets


In [2]:
OFFLINE_PACKAGES = [
    f'transformers=={TRANSFORMERS_VERSION}',
    'pycocotools>=2.0.7',
    'scipy>=1.11',
]
cmd = [sys.executable, '-m', 'pip', 'download',
       '--dest', str(ASSET_ROOT / 'wheels'), *OFFLINE_PACKAGES]
subprocess.run(cmd, check=True)
wheel_files = sorted((ASSET_ROOT / 'wheels').glob('*'))
if not wheel_files:
    raise RuntimeError('pip download produced no offline packages')
print('Downloaded packages:', len(wheel_files))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.7/411.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface-hub'], check=True)
    from huggingface_hub import snapshot_download

MODEL_DIR = ASSET_ROOT / 'models' / MODEL_FOLDER
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(MODEL_DIR),
    allow_patterns=['*.json', '*.safetensors', '*.bin', '*.txt'],
)
required_model_files = [MODEL_DIR / 'config.json', MODEL_DIR / 'model.safetensors']
missing_model_files = [str(path) for path in required_model_files if not path.is_file()]
if missing_model_files:
    raise RuntimeError(f'Incomplete Hugging Face snapshot: {missing_model_files}')
print('Model:', MODEL_ID)
print('Model directory:', MODEL_DIR)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Model: PekingU/rtdetr_r50vd_coco_o365
Model directory: /kaggle/working/uav_detr_offline_assets/models/rtdetr_r50vd_coco_o365


In [4]:
EVALUATOR_GZIP_B64 = 'H4sIAH+JimoC/719/Y/kxpXY7/1XlCkEYq84vd3jHVlqu41b7a7szUm7i921ccCkweGw2dP0dJN9JHtmWqMBfDAuQZCflAsQGJfkrBMMw2cL5+QCHLKLg38YRf/H6C/J+6hPkt0zs3JOsGebxapXr169el/1quh53qOTaL6KqkQ8fPTyuThMpnmR3I2mVVKIn6blwyLPEjFNs2SnWmVpdtTrdF7O0lIUqyyDKssin6zipBTVaS7ifLGcJ4skq6JiLRZJVaSxOCry1bIcdjqDnrgvnq2rWZ6JZV5UIp+KapbAP9M0TqO57m63P3hvB7DZqfJ8fpxWEhLAEELcfxbA//f6+Pd7e/D3+YD+9PlvX0TZBH7t9fu9zm5PvIQOAMefJXH1dikePH3wVBwe5mcKZCDSLJ6vJjAwUS6i+fzuIpmkq8XdeVQcJdCFBEejtlD9+P7Lj+5/IBSCQI+TJJsA5SYiS84qeAFDg9JpOk96grHAIU/z+Tw/LTsHBwmQHcbYWxwcUB9Iiekqi6s0z0rAShwcrKp0XuLrit4WSZwXE+jhR9AjkHoB/5TYT9IpowUOE1CX7eMog6kUiZzbiQCaL6J4BvNYitMU5mBVyUH0Op7ndTrTIl+IMJyuqlWRhKFIF4RwlGV5FRHQTkeVFUfLqCgT9RznWQWDnqeHqmQWlTPrMc3Vr5+VeaZ+L+dRBcy2UM8FkCHXT+UMh6+f1qX6WaWLhLGNgZiJIhi/nCTTaDWvkA5cZxJVUTyPyjIxdVSRrpEgTOs1PQfU0yfAj1xvGVU4JlXtGTzyi2q9RPaR5Y9h4USHc2j+IvnLVZLFiaZbtlos1yIqRbbUg8mLWIJ59vgjDWMRHcleqYIqzzJZ+JeTRS9aAZMpMFAgXwEVS6RqUugB34eaD/Jsmh4F9JvAPytyWLZlXnDZx/kkmX+YF08Pcak8TCombKfTefDR/Rcvwif3P370QozEPq5B4S2TSVLCCooyL1AlOax99XSYxuvYPMZRoX6emCZVsYqPzYPbJDpFabNTLz5clernIgfiwMO48/jhLjDyo48Av/M0myRnQ5HhigA6CHoO+BlWVQKTABNUJb41ru5Fh9rvPn6IILDukBu2gVC99dIqWZQ+NH789Cfhyx8/f/Tix08/eohUypa9CCbiKPH7PRRV/d777+Lf/l638/TDDx8/eHz/o/Dj+38RggR49ODl46dPsJUPkgzlGEixQIAA63ZQXm2t14UJ+jPN0D6wwCdJNnpZrJJuh4qYl56T6CD5CUwBBWE6wRFWVIJCKuRBw5xS0Wk6qWamxixJj2YVP1PBn4FMXSZFtaYnWHXQMln4ZTKfdsXODxEO94b/FQnIlIxWDNXo6Q67PWwGY0AIJFNCEC5AU4ShxEzvCdQsl1GcMMgyLtJlFU7SAmhBQEFsIcSw2yuSMp+fJH63B00Tib2U/2GR5xU0Me1lpXJ/MOaK2B1C1V3fL45WqM+e0RsfmJ4aw8oYheEkj6FPq2UvmkxwANTE93Z2ZM872LMXoKRIRohxoATVyMZtK6hJcpLGsAx0Sw8FADzPkvmSHwIRryZRIIBl4+XK2wruMKri2U6ZfpIovGBuDfDB1sZlkkxam93b3dxOswP2TlbGTjxL4uNlDiDkmsb/DJF0URuxxF2UACC0Sg9/ToE9DvP8OJyAYg+BDbKkCvf6FlymE1kApl+xKkE5giBF5ZqVKJFY2YIy3VFWT2IMoVkSTXoS6M2GSpbUn3KkNMC9fpgs83jWHJ8ZmkZ/QlJsry+oSfkm+O/2+3/yMQDMWw8C2rzRKIrodKcCjSVX4m3R1y8Rd5C2nl1gma3vs9mKHcF6vWGt24wjzuP82wxED4DWDJgJUQXWoZ4Tz8aUHrCvazEExMCeXK5gPGnRLueM0L3ZQOcpmLf18aGsaQzvCaBaZyDt0+TZfM2mdVqUlXjC2q8kXorA4s+PE4FDvB03Qes42UmzaVKggWehGZHVNPJKME2SEMRJ0mDu58lRkpEJYlns4uVfvCQ9XILNnmTidAZ/IgHDSCdgzMPSE8lZWlY1tpeaVeJrK1CpU1FShwCxALcLzCkfn0mNk4ZF0rFOZdu7h6+pDoMHM6b9BVmlvUWUraJ5WHuXTuVrVEa9tAyjkyido0nsd41NYFWxwITggUlQjL/U5yFrP79AkxpMhglZKjQGBsTvhwoBXU+MRoIVZcMcsRv6HmLibcVdgPBKwJTVmpWbgrXgQNJda2rwix5yMKHDXaHPB67VTWhVRCn0/HyVoUPyqCjywvce/OThfXEKvoTuLxCHq4q9zmfrl+Q2HK7SObIP+nCo6QS26nkO7zB2ktzzPJqE8zyG6SBp7RuZPyRTS2IVkzcBQzeuRQ/djxA4GjwQkNkTq2kgGCTxd4hLUlqokkDgLYIYKvxY+igeoGpBCg+j+PgQBZJFE67b21ATMPswgumyB7rFy2ngbrqxBsFdjhSWrWOSa5OpSeQP2dYJUZGhHttA2ECYpcmjBK/8ec0oiVcF2qvIN2TsvF3C9IF6XqQZyIY0NnbKYQSGNvTYI9+erOYWScAzQAgBxbZMvhSLhtDkW6TTlKxlqt2johCYF9aJ9Z4FSXLqNsmy3kcAKCr8VqC9NAunSYShiBIonWSOtybeEdI+BTAw9KqXoSabh77bT++U3JZALJIoG4H3BUSuJiOJLjMQNg+h1IX3SVLkZQPcYRqVXekeVTO5crM8PCqiib1eW7HY3xmModPlOtwwZFOtuwkSInAtHFXJmtxNc4MT4fTArJID14JQKVLQRmp2+S+WrsNlXqa4bMJkcZhMMHZW6lk27YCWj9R7/7vo0lpwe7opmASLzZOpqt96Jq+fIRf0/tBBL1strNEpgts1uJlN5GtIJIkt29uTw/hjl4Z6QLD2KjyJuFxrq6JZPZ3sUm2orGIWzUpUYxesjJFQMZBmpQRWQ5wnFP1EATQS/d6eEt9bWawHhqFeyuI7o9bFXFd190swZ5B8Utlh+FRJNGGx72JVYqzxJBGDPpp0CcaaQa8CSJCROzlJecGmaVnTe4S1rfbI0blOTEseurHINORpTvJ3aJbrY/8pWK5SyU+9R2dLGAGYMVBRyHaBOAJtfr4J7IUa5/+XmbGw029IWb2cJbav9uaTpMH+qRm6Zd7LWbS7966PkV05vU7sapKCq4DMLsPZPVnfkjDYtJcvgXJeceh1Mbg7A8tunhjCoasRz1bZMcYOU+Axfx4tDifRUNbsFQkIp/fEHaDP7j35TxesOc+2dww+vdUSQ9Q+wXRNOX4/S874l/YCDFcifx8lxbKA303mxtGjQ8LdsoRDUWPFS5Qf3yujKZgeWZkXpWcjcW4cJSSOR5a61ZUJ0XWNZ+TJvkKmMDZiUsvytqqHa3DcoKZ87pVVBCOGf0KManGDC3uBc+CTd1FKHz3pELcjlP1FHicZYOJT8oyIHHMwrPatGOp4KD3ENcIE2iCIHv62QNKUhrgn4oN/mKP4H3mrarrzntdVM0ZY6IA6/mf14i4uFbFF/9fHoPO+l068sUUUJ4o7QoJzNV3WqE0BXgsgPTdqcdzXqsYFTr2uw+lYDRkd+ALkli/pBAiT3+2NA3GcrEe8AkSRnxLBffghx8TQxjZHKVqBQKM5wt02nB92yOTb/SG9G6tYsvasQ/KeQ/KkfWXGqneTtJDTT28SKW5DcEnTKaygIS2HwJ6yod7Xcdgi6BC7gFU+lypCgghxFcA8u73SOpI1esgynpLZ6Ko5bdEtxHl0PEKmjPFxqmI9tIMQ4FC7vOmCvJY/k7M4WVbCJwj/9sXTJw9hmBOW/IF4+oJ+bEMIRiLRAM3SpOvmhrIAAwF+g2ZT75yngfYNLnrVWeV1DYGIA7kC8qCcMe2OYcgbeJK2bTZNo0DhYSIkajdvv7W2FAbYK0gf3OjMhNz2Ya+FuuoyUIta6zQBv5xe7nPDoQTwDlVVXFysslBHmZh57wRG45k9G6swUFsecnfvRrxrpExtRWxeK21LREZhnIAIvyHah5q2crVMgTclL7gd9RbH8NeX2zPkWQccAAvzYyt4wKqoyn3uq9vDDW+pnzE0EC2WsAxaAzDsFCPRMUpEe729ZVJMwc5dZailux09uYy9lkMZ7bcaGY06wlfcJWsF1pBB0wncORpNvcdqNqUBR1N44QViBd7LyKM2jJrFLjJmCbqCZo3tDT1dsCzkmrC21cAWAle/8r3nP/rAq68LZzhj002GFhiJKsk83Ek54n8CuTZDqfZH3rLyus3m5yDhhxi6XCVmagIQbFl4CLbyMUobmkJCCyoHXJnsIwJidlZ1uBDTMVizlzrixni4unJ/XxJjJj1G+Uj6bbyVEIFklJEMiTUVHLBTKJMdcBvW6ZhRwlBjHAE7MIyQwtbMcACenrgisf7g3a6r6KeKa51i0nMmyQKs/flcPvrdFjQt31dLD/JPfGBEawyudSktcOXx+3fu8GQYuHaCicUlPXB2Qa/wY8j2PG7DcWzNb+skENUM7MAZONTszdvzO7IfrOE5dgZPWmCH0GFCP0mXvjOpdoWyZk4f5mdJ6WjmfY/KvDHIrgrMBjAnaZ7g33i5gr+UweG7k1ZCRw04XHg7QNqztwFxoQ1oMwDMDipU9sERWmD+DiMCVleaTXBbApWZV+uXkoJAuoxde1JlPiBhCbRLPvzvbBCINfz/bBf+3QUYRL99ajZu1KY1iPwVnfk060yTs12xA5C63UYDXsPNFmtssW5rASuIe/kBhij6uBUugXBBcwgymFyl2SppvHxLPK0np4kYnK+jvFiDC1oKUFBi0OsN+t/n9LkV5hn1e733ew1Yql1IrilavTy3klrkbjca0dz0oiVI/JpToKfJO5dkHHSHvXvTi0A+r/UzJ3Pwb5nJQQ/edoDMOgq9Ye99bG+N4iLYGcD/mlDcabmpFXdagNRno7QB0ft3mdf7GbiPPhEEaeVjGfsEyL68OeOBgKtbtJ121Ii9Ua+R2iEFN2x6Xb14npeJsgWSebQsN5kLwJLSnOi0D7tm8LePmAzuCSxuY7IHtAyzarSLgysxPS8q4zQdkdG8acQ1A8nat9q0JdW6OZcsltWafSjfCTZIWuh9OnAo4vIELPyqSM+skEpgrP84n68WWck+9ki8RzYgCKtsEhVFJB0YcAWhBrnd9Id4cTw28imdik1uESmG6DSkOGGaie2+zk6ZHsEslMt5WhET+TUFQWBGGmIPE96WIHsL/uEFXkN9o/eGdZtyplXGkN1DspeXHD2zXaRNIu4akaQex/Uu0c9gOF3xgyaxG4i0hBk/juaUMjhB6mOf50i5i6E4V2P/TnFRGyzOkxJM3P3+sN752OEYVEslzTT6+6Wyh6CYBv/uPUrfmkXLxPdBsjSG0lWOHPAYiM6Qwogy0n6SSCSGFkO185dGBnqF0cspeAcj2qqDSZEvpSFTggMWpkcZpvaCUXGEdgSvVg5j4h4/srrpRrlC0v4pm+/YdLdy6uxik3knnaRqtQT/04JhwRvrrcJnKqFaZiQfHOAgeJ+zfJw9Piow09hsB5pAbHiEi9Eezb7zMAzE3hi9+D5vg0gxUKMJmx2gp9PFauEPUFdfBxIED9Qa3htrsVSD2UMLkKo14gUO+oFF7Y6D3CJaMmK0oef7NuUDm+BdixtXQPv3utr5OwPTJmCrIlDGBKoMF1eD4VlfGiyDgLT8WRdVg1Ht69r7de392QDfp8rDkx0TJOBSxtYsxbVbWw8NAUN1frbqA5Whgx8ilpiBsMbf65pZZFFvH96Rgdcfng1QCDOeAB7ITzEmqzJojEUJsw9MUI76FkWhOq5u+/Wg8bqjU0mPk2QZoojwWRlcs6JVzgcKpBaOsSOJmHqvyrAX1+olfiL1XmfldomD4JCXkYW7INtwPL4ZT8OQDoTfwkxdNy1ZIlFTRWeGZ9pZo1szhtdtDRzuqLc4223jO+Q5rF7nO+pjt5X31rJJg/fYoqadonxV7IAFiCdIUPOs5hHxVIZrOJrP10hxMDlK94CIPHWBkUjXvq7mbFITU+6vcT0h6vCPqy6roq3ebrPioQtwdyPEw6K1YgtItVDAZYjQgpwDmaD1DiK1Ax3W9bpT/64tMXguxB2HyqD7QYM11T2yudLTxIXdTsuq2LeUM7ZorM4xokRLhqPuxPr9ccfWqWbdOuK5G1hvjKRWyjY/SQqwJZXluEFxblS3rSJBmkVWZyQU0B3EchtWt13BGJ1RAxQ0IXRbTBkWZlUIZKcIpGpP8mJ3DHPvlu2CEGGzQrWpa0xuVS/V7ZAHi5IhhsrTtuSYHiD60SYRAiqkGVVgZBEmZVSCug8kMvtcMMQywz07NnR3MBYAG10bjJ3F6KCuff5vg/ugBffBbXAfbMR9sBl3KQlqs3CnbXyKPaQwqPHCGBq5Rd/VzCFbNPgA29QLZatVxsjJ/tQoiZ1kkRzeGChjY2sbUy2d3hsrrYf7XiorMsthWiLwIXkST2dJkfgSiO4oaGATMJ51l2GSnqSTxLfCzAY9OxlY92oK81U1UusYvJhjwMJqbG1hEoojG/Mfin7g5PKpk3RyF5mP22z3Azb5ACb2OeR41rez8xGz50lJBwnR1MAYFQc20MCzI1bG+sdcKVCoIJxDqWgtbEMVM5SSWYYkcTOmOfo6BRxbvJGorKko4U3aAd4OPYugzpRtxPm2M3b9/LTP1QuaKzUdd3XgEOcljuYxWD0VvKxKNRTBOhDaxFbmZlOZDS2+V4WuFLFDwK4kuTeuB4PHdi+OYhu2TnFdEFh9NQVEe2eSHcyM77SIFnY2g5axbqiw0ZJwZXl3w8rezJNNftzi8Ev+bNZQ/Np8UxcIt/L8n68yYyEfFUkykdYzHko9naVzzJFYlfgU4ZrAQ9orOhLsMJ3hNzNKUmj1odtBgIldVZtMDrdKcnRdi/I9a3fVVNXbDbyJ3grHYspDzGdQoxjVpICuQOGNnYFbSFhjGqPjpcEgW1BokqBb885kXraGamruK5CNsF2tzcjOtdsaM4S21hAwLsMCpQ5vpwXgISjcY3fPSJNPsed+bT4CsXEMqvEPnKm44Thqsyd/NatweEqi0HytaD5oJWqf3RVr+l3ytdDJ8HSdEmOXkZJ5A1YNlMUHcigmhHKz7ga2uLTVISwjDouGIFnjY99v6seJxa9S1p3kcRgtMUcBnGxX9aFkSMsU8+FqzpWVpbGASrzSYUpRn2UYsvD3wS4f01Y7gA3E/gAepd+/ALibW+hOoVHfNHK2Hs1CRFgYK9sNBMaHdwbWMsR3+5ps6CxbJYH1GrfZpCcQzxC0FF1TUMZZnqEE83GY+4MhRT3p9xDz6fUGnYpBUsgemmJQi9tIiGhG288UDECHndBQlbp1Q0VJ8ZA33v0bp+6YSxpulr9jaRfaY7HUSiDqJVLTwMyGruFUr2iCaVjXNqU213TzMSitRu/cU8qM94yJ07wlROaK6NwZ0treZpulvkXlEm3TtmS7pdXY7rrRFue1BjPacDfZergOjs56aUuCaUkYeUuU0UlyHwhCjgQsHzD6y3xVxAmnN4AcBY5hXEYD0jdcPrDK+yYmxyUhrKmj0vYCm1ae3AWgKCtIgHYaqaoOXJOVU+NMFe1ybD6ntqGUqmuHpdwsxNAlstu8bkKepOUEedMsZbyA40YLuX0Rty7gTi1RGyy3D6O0mk1X8/r9OgcH6Gjcj0ExRvGafUS6JmSWmBtrpC9pLEAadqUHa/l/dRGlF6uLfR1vJmptM4avsNgYPIuKltrAB60NOia2rAUFhokFz28Jpi9fh4GuFUkdOgonHWW0T6I0o1tcqp6EBYKnTIqTBGgVVaDnZ9FJSvvLQLsCCzNRgnGdVfO1KE9TaXBXOXS1iOIC/oUeMIWAUweYRSghma+jkWIRjOGaPOTzGlr/sVAkJcjUIrHYkpfCYLU8pGfnyKSTgGL3YmeesB2MHTMXDBtb3ODuZWv/qDLbeBbkbtP6aw5drToLByu9i0tr0hsM6f0NXQZiuDdu4D1ulSScroGXgIDOJn+36lpNm1272zhgV+CCUIaxfEzMpU1mH2fDNS81+kgDLTxch5YTaLIfbD3sHBbQ+Y488pAupnGuoXGqNlO75FCdzVOVQtckg5YFLfNLifegojbFIoa0R9QIo7vE6zbTxRgLB2qzq/3WHKaWeps4p9G+icifNlrWHCJOQn2w1yRVqVnUTKNYUhe43OiyRrc9DS4MXIf+htG6G5BrY90GGTfW1NG+jTXYHtcE6LbX7LaWtq3D/Rpxx9pYMI5Vp3PtjNA5rZpwaOuuZVLq4RXbhWKwXfJ6uQfydKEWpWz5eARhtz1DkpKUcDmacdwyLxLU0XEt0dTyY4e897Ml5VTBoLwB9t+t9vgmqG+f8vbtKtHb74omnFTgQhupA+JuEgC6pxgocsHovdWmomgCiYp9S2sFzel2VMS4kSbuMivvQkl/vNulw3+YtiqDsbKcZxbKW4RCS8rq40y4Rh9eY5iWJuQ8EXy9IxqB0P3DpCrFPM+XQQswNHboAgOwjDi7lW9TycD0kG1He3jrIuqostfGbjUtCZOz19+QiDvFjMYbznC/fSXrkELbRLftKdb/A0aoo/FODdItBEu03MovyCBOUMaKiWh26Nhnw1oMya0XhTzJZRNYhQ1jUZzC7GI2RjbRx4YzsNsV6VVKQLPTZvoOL7FyNUfzyjoFev+ZN7TYPYkyH4giu8GtQOcgKF7tubV+v17/e3tb6+/V6j8fNKsXFjrNHp4P+tc1GTSbXNtmt95m7/o239VtLjacuUXbH6B4ywSc96wK+yGiYvUjLz6FOjxZ1qs0Xxl1hDX2SSD6dm4qKhU3P9U1LcZ2V87C99iu3WgY24joa2SxJQ6n5tZq13XDhaxey/FfOp1rnQa5mVe+2f8m+xzfjesn2JRrRy83xbpUxx3LBMG9v28dXNqQsxhwxCZw7V8w+ch14K5coWwNaOMJhPNWoeepE8vEZYSpKmk3yjwLKY+PBNs+JSVVtTfEy3yRU+UZiK4+IqJ/qfxOfpLpUuMN4IhGehXSU4sdedFypkBdxWWIVg8OEQOWq6W5ItmXlynVTqK/OU+amJBzIJjvr11jR7hAcAsv1nfH4o2I22viCOza+GyfEn5Mr0jf4AUICf7YqpTcqyOs7tBMwQut0kLe9+eQi87Gqpube8I92eE9zsDUBPsOsbuLl9ndlZCwdYmLxL5cgkdKqHbMdQ7WITukcYEn2cpqkq8qP817L6Dj7OjxU7/rXEQFs0d7VEgany47UBNqmd9UNsFqsgGdy36elH67cGo5v0aBNANR8hWl2ahZ8SXwQPUXCBDqL+kQIq2UluZ4bVy0KGGFHj3mGwlqS7ZdcI03Q1I25YgFfssltm1oqHXit72MwJhdkPHa+hrsw0VUAHfIt/pchVUFlAlwQU4oyQuH8F5ZJ6oCKmvpKMto6V4iSiVgcdRK6OrweiHfJF4vpYvF3cIiHDQK+s2SRlFLr0Vrr4XTq0zBiE7NXceN0ydYbB3LxaAQUUuWld3r7A88ONKvjctYHtFpszyUJgu8l1hx98onakEL4Oijwv/qdkwbW7tmB2ZaYBhIDtDnm+DkgXn2wfihJru5XVqSD3N+oY0HHr28T07jzc+KDtYtEHSfEVcb8dM+QhhbMkn1wy8cu521K8GGEduduvrQI8hQxeqvXuOwzOcr1IC08+nWxdznzcCLBNY8uFyyqcUjTVvEt6Hedem0gztMxqFq8VKd6t+xM0OcY9iYquhqf4OxsyYMcfWppeJ4kp9m6kJAxpaiI76prLhjkh6llT4qd8+9mEif2DU8+qn4mD+38Kl4WXz9h6vXfxub+5ig8EW0sp71Nbrw5sHs8h+ymZhfvf4P9Pzyq1/gPsbXf/j6c/j3q8+uXn+Wik/thfzpzs4O/n9Y/+MIGMOx8lCZGWSTWdU8o/mLd8Fsmvdxo0UoT+N72V2+wFODcq6JIZuZXwzf6Z0zeZEtLqYX/8brXH/edgokJrl0ATQ6RyTf5ol8ezxU8BCWfkvz23zp1aDKyu4SgWbvWO1gFtKr179YCIZujxywqV+ZJRmwfmZWsiEfOi0S2piU1/MopuPSsHYflrklVu/+ORJK3/FjSr2xxZrNTVG11kq6A1rNrcU7uianBjW73cewQr0uBbUsfNjBLQEZFlLmxmkqa+CFhUpJjaWBWxVRqAOHblf70mfHbI/mG9TXY3mNWKHzw70KVxfxqTXG/YaIHIsf6uQl7yi9evXFwuvonY0QLwB1Qovey/zyV6ALrl7/EiTdu4O+gCawqptXUdPxWUkgvAm6TCoct4zsY9DTo22SAQDpOKJv6r3Q9yXjCVEG8rYE8vZ4/20LyNvjC0ZBnUZW98ZWfH/g1PO8t8QHl5/nIsY/X312+Tnge5Refs4n6asCxJIOWHU6b70lns2uXn2+ECdpp7MjXl69+nJpNxuKg1akcM0CNgfim5//F3Fu6HfRQygNWQlr4er1/wTJN4fOUnPtYHX1+nfiQF0zTw7GDl8zv7PXP8DAXrSm6+HhF0zFDG+7A5n6t0vxjrnmLhCLq9d/JwqYJugBv1bzzb//z/blfuIEmqR0Was9HL46HUdTu00Wi7AyDhDHs0HMD9uB1VcFlpmb2hTMP59d/m/A9vLzJZLm1/Dzyccverg/rm7e+OozpOIXsSivXv3jUpxdvfrjUjDXQpNXv8kwaJzzBboTdGgRbFv7s9XVq19XOKuvoBs1+YD9WbAOToNZwDEMGVGd8AUEB3L3TAy++fnfDPo94pU/BxwqoCrgIOLZ5Ze4FmZXr38bWyz14PLzmDmughn5T2JCvJBi93+/FvPLX4nljLEvvvrFoic+wNUlv2pUrS7/Xg5LJsrICJQeC8eogG+QjeTneqxZ0J4JEl2GrUKuhoQfYtB+cvl/gAwHVpz9AL9yJMsHdwd9+F//7h5erqpOaXCikEXSq9dfwlj+BQYhWf14lpoE4RMYpenpp08fREuY9M75NmulRR6DP9DZwHv4oabaFJxrkXjB6+rOnXNbHFqa9R5ozTt3YAAgDOq1tIblSrjVZpsyjeotOvaepVztyb5zh9kIWWQmZlhhJacehc9DItchCrgj6MeE1e1hAtpI6hI4hIQUL+2YFhNIt99D0Sy/evXPyINRXmdPOSXqnEd2+as1rpoD0jkHAOzV7zIBTb5Y0ngldBKdljz5PjT8Jw1Kqwjr1hy+yjSGWojAL0S8gqoxLMNlgHPzGS6HCAaS82ARNotmkGT/NWUZz7R6AvT7MhNnl7+txLFZfp3NYinAjr4Q31OLqp2OZIP+1UrEQKQZGKWWTG5I7x4yW8UW7GoNdKmUBfttuWFHXL3+H/ZNmA6D7/VJXjidkCXwNr6zGDmbff0HQA63LTa3+N6eadED+StpHxMzIicAHsQwr369lhNYXf5+YRY9zM0vK5DGrz9DRvmyknOLuudx/hMU438dKFSQPr+ECb6EaofoJ6NYwE+kMYvFl/8rI0wVYWKiC05cirfNA40zpA6IJUn3xoKWw0KTyKIEIoMaUHvWLYtctdxzWuKq2sLSKAeuXv9H1MP/sEB4thU3dCaZ6PvTy98TldZqzlFt49p6BbwGoF79UW0dH7MmZPBAt1d/zHi4oOtw9XGTKl/uoJsJRCENAzX+mddo1eZaFaTyuM8TQEVz9xEh0/8+jVcKjEOcfGz4N6CPM+AFoNjXf1iJKsVfrvyrLQDSVW3czav3R/YQ4qvXv4kMJt/8/L81l9o3P//vnc4HqookDcIFvmD+sxYLqNQvDHvOCWUk3JyZCXsFpv6/v+2JJyCJdMXqEnSntMdQRJFdRA3pOz7mAnQe3Tw/AimibscXzJysgYc0R/8o0DYDRactMlCoxzCdf5dStRwxByqA7cYVrfuKUVMyS+1K+boWGczR71awkpD8zEloAt5teYukef1rgLAC2sTwRIWkQjhTkeF//QewbxHtgJevJetyXJfaYDYSG0gPXMb2PlCA6cMz4PSi6UJ0/hjw+CxjCitCuQzDIBAiNJDLnmbOmlepwCx9wqwjBbgzvcAD/ySNM+QSkfFqQATmaMx3OgcHB8v8NCnKWTKfd5a852fi+XonhT4fpNy33nIt1OexBH0Uz/66leWoFLwFQVa0vpsRbC3sFpUUXnNKq/o3vMR+A3anvZlAziTd3XRQs1+rFi/CamrF9e/W7Pe7dVDlRgO+HV7dhGeAL6XVAE4JCGyY3ld/XDkgNn1qE5uTNACRA3P4CoaGAiMDJotSE03vdD6+hBm1F5gWjsCXbArffRpX0QnGa7WANew7MRYU7+rKHV6oseDFbH0hMxD1NGYoabnfJhDWaVhsBPbp0wgfaHWxeYtPvJik4U6yia11/hYmB1F0KMS+p0s6sM07t1R4xbXkffWo9+ncQB75iyNh1cJ7wnpHaUUfX/rxo/sPnYuB6fNjm+8D9lbZcQZGu2c2QfD2cmy05cZfda+V6ohv7qS7zDCveeZ7RTIdCs/pb6ruNd6AOwHZ3xuOnStaZLO2ITgXPk+vvabYQVq2oz51tDXN/Fr4CnPVKD5kfSZJ3VUMTz0jFcQPtl5973w/jy+4P0zwM63zJILfA5VLU/vwIHVil5m70Dvmckz5YUOqbEo2Vr3pVbl6U1LBpgKM5dRQwVkMrc++UkX7jmp98awEhFs9G+DIW79bDindoLGFgye/OmVYbWRTy/28miPQ1GqWe8p2zJznnOXxdTfcB24rlrpv1AhF9c0b6lkL3NuA6ZCFAe8evrDu8XPWphaoTpx+kZZ0OBhIg5vXSxJSGLynBU4bvpJ4UgjR0uSvkfnm8LgEU181H8Iyf5JXH6IJJRfPx7JDA5fOkgzxPsd3rLC1hNjtdmpX5m+7zT9gWlOYr/aZrtq3xKievDdarlbMO5l6D+Ut1uf8EuP98h7kc5mUSbeKY7mRGKPzmgi50DJAK2slB1zOtSrIS6lX4P9gIpDnadHUcvH91Aup9NxUuPCsLDwloJ3epioofs69XFgL21J5bc3wpW61mMijQsaeUZ+VcL4X4Xxvon3BWR+E6Jh9y5sDrK/Fdni47G4L012qdbio4WHBqVs63aRHFVnnr19sk3Ru3htnXPCe9Y2yMKzW+kJmu9ct1GlrSqyLO/BeOcMvOAC6yYS2bN9/tx8Ib57j0VVdOHi337d33vWuO7xz7urxsgUOim4stUr5C5SYL6BDP2fr05kK6OINu3S7biC/JyEO11YU2bNTIfk69XKZxKU7FTWLGwfn7jFbs292uq3C2pa08sjog4W4l41kxt9uvTt3agxi7RNbFHC/OkSUd/FpLIegCboJ0Nrpuma8rcvjtkO56LhfHZBfBnQ/NL75K3vXcOr2b+1x0rK65rzET+Zh7hzKVFRoMvFvbFIpzMX7yCl+g0GCxqwELWQ1J9UNPMqkNNCHLTenS+OloRUoq9P6JICStJ6VH6oFjcXr+6aNY2qjoiay0rdKzVck6Ozutq+jtKMcCHMbsRQ9Zte5njbKavR5gl+cJPgTNzn23IVdv1m2MZn2GOnTY25uRuOUSjqtf05CWSsupuVsVaXzXrGoiiTx206s2t/R4hnG25IbEsU90ie/Dbbt64/XacLASJWue99EDZW6+GjFZPPnzW6kQNUG8LcAvl2T3mTSWz6F4mDCXz4yP4NmlZH1fRTDrFI4jWpfTLEODSGzj1Q+ZsuHmYBbRi1GeXMRjWprqnYSlNfXqClkjRU7sr+poq9BMUZozQZt+0gTzpr8BpqxFR15qdLWpbi0zn7cQLj+KwrE7R9v0ecG1QmB0bXH9G91mF15ZzbclgxvyzHZlNLbaTko3b4Irr913prSWtuaEdBMfhlqUtUMnuaooC4W2plvbioOizb2uQiXhsy8Wf5NI7dHw6sJvjcBh1KpCfLWOUJEnhsNuIWQBlotxY3q3mDUbwbzJkO/DWTzOTzL8FafOQf1UIWrKgauwQ8XEhNn+amPPz7BAD68w4945ewN2De6WX5UjX/po3ZDJ+7zPsd9VMZTPZMU3C8ULLdzycwCsKNR8puGOs2/Vt/OpRoKO2bQ7gNwQk5zjNd5LoycVqv2yLZ52c3zLN4kmdKpbT5sRXtrlElAhyIK8xVqkLaD/jsD+/OeaE9aCQdeC/TtjhJ7GNe5XRdtWcdvRJobecQGsab/dQO8trhf12K20VbajN1W7+2inen0Xk6T71To1FpeGzeM6stMhRytb39aUcjuhtq8W+INt2yf1JtqWS639RDbadstOIdJHKEn4mxKYShtlUUnUTqn4+k14I7Ys3uwj0apQzdeO4FNq/P6eSdEDomzLnsnSVE6N9lylXlUoTSESupnT/3wm0Qs4pmnvroXhhJkGNbrFVFW0nc+CmTLMOQjXWHou6+6W2Cw/SknVgZPW6uEUkA30/Stj8vgV76s6grgxi/UcODTUER/p667KSVfRmyUQaykUJuJzKK77obQacRaWa1N02yDRs3CDVwic5TVcVyrmp1PPbzWfrOORtTPndTMsqBmB3Wvtfe2wLasn8AyMOowm+ju9ev5303gDd5xEQ9aDLqtjNAysjdCw4wzqBlVW7u/aSL8bQjRPqVvQok3xmUDC2zD4cI9xK63K+ytdutLWNK8vPWHsOSHultOPDhnHdztn+fKIz63EDO7OaoSth4KrKThXFAWAAivkMRZGJLkCkPciw5DT30XGDemO/8POD4D6HmVAAA='
evaluator_path = ASSET_ROOT / 'eval/detr/evaluate_detr_visdrone.py'
evaluator_path.write_bytes(gzip.decompress(base64.b64decode(EVALUATOR_GZIP_B64)))
print('Evaluator:', evaluator_path, 'sha256=', hashlib.sha256(evaluator_path.read_bytes()).hexdigest())

Evaluator: /kaggle/working/uav_detr_offline_assets/eval/detr/evaluate_detr_visdrone.py sha256= 4e698495657688a9fc3854557187067f754e82934c8b37f19ebeb8c08883c936


In [5]:
# Validate the downloaded Hugging Face snapshot without requiring a separate checkpoint.
config = json.loads((MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
weights_hash = hashlib.sha256()
with (MODEL_DIR / 'model.safetensors').open('rb') as weights_file:
    while weights_chunk := weights_file.read(1 << 20):
        weights_hash.update(weights_chunk)
MODEL_AUDIT = {
    'model_id': MODEL_ID,
    'model_type': config.get('model_type'),
    'architecture': config.get('architectures'),
    'num_queries': config.get('num_queries'),
    'anchor_image_size': config.get('anchor_image_size'),
    'weights_bytes': (MODEL_DIR / 'model.safetensors').stat().st_size,
    'weights_sha256': weights_hash.hexdigest(),
}
if MODEL_AUDIT['model_type'] != 'rt_detr':
    raise RuntimeError(f'Unexpected model type: {MODEL_AUDIT}')
print(json.dumps(MODEL_AUDIT, indent=2))


{
  "model_id": "PekingU/rtdetr_r50vd_coco_o365",
  "model_type": "rt_detr",
  "architecture": [
    "RTDetrForObjectDetection"
  ],
  "num_queries": 300,
  "anchor_image_size": null,
  "weights_bytes": 172175856,
  "weights_sha256": "59abc0b00bea0cc8f7e9ce053c29189a348605bb337123609a7fe02e4ea830a0"
}


In [6]:
if COPY_VISDRONE_FROM:
    source = Path(COPY_VISDRONE_FROM)
    target = ASSET_ROOT / 'visdrone'
    if not source.exists():
        raise FileNotFoundError(source)
    shutil.copytree(source, target, dirs_exist_ok=True)
    print('Copied VisDrone to', target)
else:
    print('VisDrone was not copied; attach the raw dataset separately to the training notebook.')

VisDrone was not copied; attach the raw dataset separately to the training notebook.


In [7]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()

manifest = {
    'asset_format': 3,
    'model_id': MODEL_ID,
    'model_folder': MODEL_FOLDER,
    'transformers_version': TRANSFORMERS_VERSION,
    'model_audit': MODEL_AUDIT,
    'offline_packages': OFFLINE_PACKAGES,
    'files': {},
}
for path in sorted(ASSET_ROOT.rglob('*')):
    if path.is_file():
        manifest['files'][str(path.relative_to(ASSET_ROOT))] = {
            'bytes': path.stat().st_size,
            'sha256': sha256(path),
        }
(ASSET_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Files:', len(manifest['files']))
print('Size (GiB):', round(sum(x['bytes'] for x in manifest['files'].values()) / 2**30, 3))
required_outputs = [
    ASSET_ROOT / f'models/{MODEL_FOLDER}/config.json',
    ASSET_ROOT / f'models/{MODEL_FOLDER}/model.safetensors',
    ASSET_ROOT / 'eval/detr/evaluate_detr_visdrone.py',
    ASSET_ROOT / 'manifest.json',
]
missing_outputs = [str(path) for path in required_outputs if not path.is_file()]
if missing_outputs:
    raise RuntimeError(f'Offline asset bundle is incomplete: {missing_outputs}')
print('Complete. Save Version, then attach this notebook output to the training notebook.')

Files: 38
Size (GiB): 0.233
Complete. Save Version, then attach this notebook output to the training notebook.
